In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_squared_error

import matplotlib.pyplot as plt

import os

#Running on Google Colab = True/locally = False
colab = True

### Get the File Location of the Dataset

In [ ]:
dataset_path = os.getcwd()
# print(os.listdir(dataset_path))
if colab:
  dataset_path = os.path.join(dataset_path, os.listdir(dataset_path)[1])
else:
  dataset_path = os.path.join(dataset_path, "Final_Dataset")
  dataset_path = os.path.join(dataset_path, os.listdir(dataset_path)[0])
print(dataset_path)

data = pd.read_csv(dataset_path)
# print(data)
X = data.iloc[:, 0:data.shape[1]-1].values
# print(X)
y = data.iloc[:, data.shape[1]-1].values

/content/concatenated_data.csv


### Create the SVR Model

#### Resources:
1. https://www.analyticsvidhya.com/blog/2020/03/support-vector-regression-tutorial-for-machine-learning/
2. https://scikit-learn.org/1.5/modules/generated/sklearn.svm.SVR.html
3. Picking which kernel should be used for linear and non-linear relationships: https://www.geeksforgeeks.org/support-vector-regression-svr-using-linear-and-non-linear-kernels-in-scikit-learn/
4. https://www.geeksforgeeks.org/time-series-forecasting-with-support-vector-regression/
5. https://www.geeksforgeeks.org/ml-feature-scaling-part-2/

Note: Our dataset may need to use a kernel that is applicable for non-linear relationships since time-series data is prone to alot of changes.

### Creating the Train-Test Split

#### Resources:
1. https://www.geeksforgeeks.org/how-to-generate-a-train-test-split-based-on-a-group-id/#importance-of-groupbased-splitting

### Generating the Metrics:

#### Resources:
1. https://scikit-learn.org/1.5/modules/model_evaluation.html#r2-score-the-coefficient-of-determination
2. https://scikit-learn.org/1.5/modules/model_evaluation.html#mean-squared-error

In [ ]:
##Without Using MinMaxScalar

##Default Hyperparmeters: C = 1
svr_model = SVR(kernel = 'rbf')
##Overall there are 25 groups since there are 25 patient IDs
kfold_split = GroupKFold(n_splits = 5)
groups = X[:, 0]
r2_scores, rmse_vals = [], []
# Apply splits while ensuring patient grouping is maintained
split_num = 1
for train_index, test_index in kfold_split.split(data, groups = groups):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]
    print(f"Unique classes in TRAIN: {np.unique(X_train[:,0])}")
    print(f"Unique classes in TEST: {np.unique(X_test[:,0])}")
    # print(f"TRAIN shapes: X: {X_train.shape}, y: {y_train.shape}")
    # print(f"TEST shapes: X: {X_test.shape}, y: {y_test.shape}")

    X_train, X_test = X_train[:, 1:], X_test[:, 1:]

    #Fit the model
    svr_model.fit(X_train, y_train)

    # Predict
    y_pred = svr_model.predict(X_test)
    print(y_pred)

    # Calculate r2
    r2 = r2_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    print(f"Split Num: {split_num}")
    print(f"R2 Score: {r2}")
    print(f"RMSE: {np.sqrt(mse)}")
    r2_scores.append(r2)
    rmse_vals.append(np.sqrt(mse))
    split_num += 1

Unique classes in TRAIN: [ 2.  3.  4.  5.  6.  9. 10. 11. 12. 13. 15. 16. 18. 19. 20. 21. 22. 23.
 24. 25.]
Unique classes in TEST: [ 1.  7.  8. 14. 17.]


In [ ]:
r2_scores, rmse_vals = np.array(r2_scores), np.array(rmse_vals)
avg_r2 = np.mean(r2_scores)
avg_rmse = np.mean(rmse_vals)
print(f"Average R2 Score: {avg_r2}")
print(f"Average RMSE: {avg_rmse}")

In [4]:
ts_split = GroupKFold(n_splits = 5)
groups = X[:, 0]
print(groups)

param_grid = {'C': [50, 100, 1000],
              'gamma': [0.1, 0.01, 0.001, 0.0001],
              'kernel': ['rbf']}

grid_search = RandomizedSearchCV(SVR(), param_grid, n_iter = 25, scoring = 'r2', cv = ts_split, refit = True, verbose = 3)

grid_search.fit(X, y, groups = groups)

print(grid_search.best_params_)

[ 1.  1.  1. ... 25. 25. 25.]
Fitting 5 folds for each of 12 candidates, totalling 60 fits


/usr/local/lib/python3.10/dist-packages/sklearn/model_selection/_search.py:320: UserWarning: The total space of parameters 12 is smaller than n_iter=25. Running 12 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


[CV 1/5] END ......C=50, gamma=0.1, kernel=rbf;, score=-1.271 total time=  30.7s
[CV 2/5] END ......C=50, gamma=0.1, kernel=rbf;, score=-3.417 total time=  33.8s
[CV 3/5] END ......C=50, gamma=0.1, kernel=rbf;, score=-4.457 total time= 7.9min
[CV 4/5] END ......C=50, gamma=0.1, kernel=rbf;, score=-3.204 total time= 7.6min
[CV 5/5] END ......C=50, gamma=0.1, kernel=rbf;, score=-0.001 total time=   0.0s
[CV 1/5] END .....C=50, gamma=0.01, kernel=rbf;, score=-1.036 total time=   9.6s
[CV 2/5] END .....C=50, gamma=0.01, kernel=rbf;, score=-1.512 total time=   5.0s
[CV 3/5] END .....C=50, gamma=0.01, kernel=rbf;, score=-3.235 total time=   5.7s
[CV 4/5] END .....C=50, gamma=0.01, kernel=rbf;, score=-0.518 total time=   8.8s
[CV 5/5] END .....C=50, gamma=0.01, kernel=rbf;, score=-0.001 total time=   0.0s
[CV 1/5] END ....C=50, gamma=0.001, kernel=rbf;, score=-2.483 total time=  25.2s
[CV 2/5] END ....C=50, gamma=0.001, kernel=rbf;, score=-1.476 total time=  19.9s
[CV 3/5] END ....C=50, gamma

In [5]:
##Using MinMaxScalar for Standardization
ts_split = GroupKFold(n_splits = 5)
groups = X[:, 0]

svr_model = SVR(kernel = 'rbf', C = 50, gamma = 0.01)
scaler = MinMaxScaler()

min_max_r2_scores, min_max_rmse_vals = [], []
# Apply splits while ensuring patient grouping is maintained
split_num = 1
for train_index, test_index in ts_split.split(data, groups = groups):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # print(f"Unique classes in TRAIN: {np.unique(X_train[:,0])}")
    # print(f"Unique classes in TEST: {np.unique(X_test[:,0])}")
    # print(f"TRAIN shapes: X: {X_train.shape}, y: {y_train.shape}")
    # print(f"TEST shapes: X: {X_test.shape}, y: {y_test.shape}")

    X_train_cyclical, X_test_cyclical = X_train[:, 1:3], X_test[:, 1:3]
    X_train_sleep, X_test_sleep = X_train[:, -1:], X_test[:, -1:]

    # print(X_train_sleep, X_train_sleep.shape)
    # print(X_test_sleep, X_test_sleep.shape)
    # print(X_train_cyclical)
    # print(X_test_cyclical)

    X_train, X_test = X_train[:, 3:X_train.shape[1] - 1], X_test[:, 3:X_test.shape[1] - 1]
    # print(X_train, X_train.shape)
    # print(X_train[:10, :])
    # print(X_test)

    scaled_X_train = scaler.fit_transform(X_train)
    scaled_X_test = scaler.transform(X_test)

    scaled_X_train = np.concatenate((X_train_cyclical, scaled_X_train, X_train_sleep), axis = 1)
    scaled_X_test = np.concatenate((X_test_cyclical, scaled_X_test, X_test_sleep), axis = 1)
    # print("X_train = ", scaled_X_train)
    # print("X_test = ", scaled_X_test)

    svr_model.fit(scaled_X_train, y_train)

    y_train_pred = svr_model.predict(scaled_X_train)
    print(y_train_pred)
    y_pred = svr_model.predict(scaled_X_test)
    print(y_pred)
    r2_train = r2_score(y_train, y_train_pred)
    r2 = r2_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    print(f"Split Num: {split_num}")
    print(f"Train R2 Score: {r2_train}")
    print(f"Test R2 Score: {r2}")
    print(f"RMSE: {np.sqrt(mse)}")
    print(f"Y_PRED: {y_pred}")
    min_max_r2_scores.append(r2)
    min_max_rmse_vals.append(np.sqrt(mse))
    split_num += 1

[0.0943958  0.09465824 0.09416278 ... 0.0853164  0.08453371 0.06326422]
[0.08798891 0.09130774 0.09077066 ... 0.08948175 0.08978923 0.08891403]
Split Num: 1
Train R2 Score: -0.937136371205499
Test R2 Score: -0.5748196915349437
RMSE: 0.03358375649374091
Y_PRED: [0.08798891 0.09130774 0.09077066 ... 0.08948175 0.08978923 0.08891403]
[0.08867395 0.09146811 0.09097769 ... 0.0859682  0.08481267 0.06328916]
[0.0685833  0.07873934 0.08587907 ... 0.05369045 0.05279079 0.05441642]
Split Num: 2
Train R2 Score: -0.5950429953312277
Test R2 Score: -2.1675179478333786
RMSE: 0.05746559620308179
Y_PRED: [0.0685833  0.07873934 0.08587907 ... 0.05369045 0.05279079 0.05441642]
[0.08439534 0.08725226 0.08674133 ... 0.05739517 0.05744194 0.05891352]
[0.08131219 0.08142446 0.08170529 ... 0.08760789 0.08675043 0.06588649]
Split Num: 3
Train R2 Score: -0.5756155140966879
Test R2 Score: -2.9387013072582
RMSE: 0.06782525740766795
Y_PRED: [0.08131219 0.08142446 0.08170529 ... 0.08760789 0.08675043 0.06588649]
[0

In [6]:
min_max_r2_scores, min_max_rmse_vals = np.array(min_max_r2_scores), np.array(min_max_rmse_vals)
min_max_avg_r2 = np.mean(min_max_r2_scores)
min_max_avg_rmse = np.mean(min_max_rmse_vals)
print(f"Average R2 Score: {min_max_avg_r2}")
print(f"Average RMSE: {min_max_avg_rmse}")

Average R2 Score: -1.550702077277221
Average RMSE: 0.055906612028092016


In [7]:
##Using MinMaxScalar for Standardization
svr_model = SVR(kernel = 'rbf', C = 50, gamma = 0.01)
scaler = StandardScaler()

##Overall there are 25 groups since there are 25 patient IDs
ts_split = GroupKFold(n_splits = 5)
groups = X[:, 0]
ss_r2_scores, ss_rmse_vals = [], []
# Apply splits while ensuring patient grouping is maintained
split_num = 1
for train_index, test_index in ts_split.split(data, groups = groups):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # print(f"Unique classes in TRAIN: {np.unique(X_train[:,0])}")
    # print(f"Unique classes in TEST: {np.unique(X_test[:,0])}")
    # print(f"TRAIN shapes: X: {X_train.shape}, y: {y_train.shape}")
    # print(f"TEST shapes: X: {X_test.shape}, y: {y_test.shape}")

    X_train_cyclical, X_test_cyclical = X_train[:, 1:3], X_test[:, 1:3]
    X_train_sleep, X_test_sleep = X_train[:, -1:], X_test[:, -1:]

    # print(X_train_sleep, X_train_sleep.shape)
    # print(X_test_sleep, X_test_sleep.shape)
    # print(X_train_cyclical)
    # print(X_test_cyclical)

    X_train, X_test = X_train[:, 3:X_train.shape[1] - 1], X_test[:, 3:X_test.shape[1] - 1]
    # print(X_train, X_train.shape)
    # print(X_train[:10, :])
    # print(X_test)

    scaled_X_train = scaler.fit_transform(X_train)
    scaled_X_test = scaler.transform(X_test)


    scaled_X_train = np.concatenate((X_train_cyclical, scaled_X_train, X_train_sleep), axis = 1)
    scaled_X_test = np.concatenate((X_test_cyclical, scaled_X_test, X_test_sleep), axis = 1)
    # print("X_train = ", scaled_X_train)
    # print("X_test = ", scaled_X_test)

    #Fit the model
    svr_model.fit(scaled_X_train, y_train)

    # Predict
    y_pred = svr_model.predict(scaled_X_test)

    print(y_pred)

    # Calculate r2
    r2 = r2_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    print(f"Split Num: {split_num}")
    print(f"R2 Score: {r2}")
    print(f"RMSE: {np.sqrt(mse)}")
    ss_r2_scores.append(r2)
    ss_rmse_vals.append(np.sqrt(mse))
    split_num += 1

Split Num: 1
R2 Score: -0.36578616990418444
RMSE: 0.03127556826258323
Split Num: 2
R2 Score: -1.4437975359632405
RMSE: 0.050475541408885556
Split Num: 3
R2 Score: -1.6668477927627752
RMSE: 0.05581025949720391
Split Num: 4
R2 Score: -1.0557757474832807
RMSE: 0.049836551050662065
Split Num: 5
R2 Score: -0.0012414832329845638
RMSE: 0.059744568030300954


In [8]:
ss_r2_scores, ss_rmse_vals = np.array(min_max_r2_scores), np.array(min_max_rmse_vals)
ss_avg_r2 = np.mean(ss_r2_scores)
ss_avg_rmse = np.mean(ss_rmse_vals)
print(f"Average R2 Score: {ss_avg_r2}")
print(f"Average RMSE: {ss_avg_rmse}")


Average R2 Score: -1.550702077277221
Average RMSE: 0.055906612028092016


### Generating the Plots:

#### Resources:
1. https://www.analyticsvidhya.com/blog/2020/03/support-vector-regression-tutorial-for-machine-learning/

In [ ]:
##Code from
## https://scikit-learn.org/0.22/auto_examples/model_selection/plot_cv_indices.html#sphx-glr-auto-examples-model-selection-plot-cv-indices-py

from matplotlib.patches import Patch
cmap_data = plt.cm.Paired
cmap_cv = plt.cm.coolwarm
n_splits = 5

def plot_cv_indices(cv, X, y, group, ax, n_splits, lw=10):
    """Create a sample plot for indices of a cross-validation object."""
    # Generate the training/testing visualizations for each CV split
    for ii, (tr, tt) in enumerate(cv.split(X=X, y=y, groups=group)):
        # Fill in indices with the training/test groups
        indices = np.array([np.nan] * len(X))
        indices[tt] = 1
        indices[tr] = 0

        # Visualize the results
        ax.scatter(range(len(indices)), [ii + .5] * len(indices),
                   c=indices, marker='_', lw=lw, cmap=cmap_cv,
                   vmin=-.2, vmax=1.2)

    # Formatting
    yticklabels = list(range(1, n_splits+1))
    ax.set(yticks=np.arange(n_splits) + .5, yticklabels=yticklabels,
           xlabel='Sample index', ylabel="Fold #",
           ylim=[n_splits, -.2], xlim=[0, len(X)])
    ax.set_title('{}'.format(type(cv).__name__), fontsize=15)
    return ax

In [ ]:
my_groups = np.array(data['PatientID'])
this_cv = GroupKFold(n_splits=n_splits)
fig, ax = plt.subplots(figsize=(6, 3))
plot_cv_indices(this_cv, X, y, my_groups, ax, n_splits)

ax.legend([Patch(color=cmap_cv(.8)), Patch(color=cmap_cv(.02))],
              ['Testing set', 'Training set'], loc=(1.02, .8))
# Make the legend fit
plt.tight_layout()
fig.subplots_adjust(right=.7)
plt.show()